# Cleanup, variable selection, and QC

In this notebook we will go through the QC procedure to select, and clean the NHANES data

## Loading data

In [1]:
import os
import pandas as pd
import clarite as cl
import numpy as np

rpy2 ModuleSpec(name='rpy2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fc33aa02c50>, origin='/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2/__init__.py', submodule_search_locations=['/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2'])


In [2]:
#### SET PATHS
os.chdir('..')
mainpath = os.getcwd()
datapath = os.path.join(mainpath, 'Data')

In [3]:
os.chdir(os.path.join(datapath, 'nh_99-06'))
nhanes = pd.read_csv('MainTable.csv').rename(columns={'SEQN':'ID'}).set_index('ID')
var_description = pd.read_csv('VarDescription.csv')\
                     .drop_duplicates()\
                     .set_index('var')
# Convert variable descriptions to a dictionary for convenience
var_descr_dict = var_description['var_desc'].to_dict()

In [4]:
os.chdir(datapath)
survey_design_discovery = pd.read_csv("weights_discovery.txt", sep="\t")\
                            .rename(columns={'SEQN':'ID'})\
                            .set_index("ID")
survey_design_discovery.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT2YR,WTINT4YR,WTMEC2YR,WTMEC4YR,WTDRD1,WTDR4YR,WTSBA2YR,...,WTSPH2YR,WTSAU4YR,WTSAF4YR,WTSAF2YR,WTSCI4YR,WTSPH4YR,WTSCI2YR,WTSVOC2Y,WTSAU2YR,WTUIO2YR
ID,,,,,,,,,,,,,,,,,,,,,
1,1,1,5,9727.078709,4291.490177,10982.898896,4456.206594,14809.893854,6066.128663,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,1,26678.636376,14203.335998,28325.384898,15336.199717,26899.708892,14921.934292,NaN,...,NaN,NaN,33073.267573,60586.147294,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2,7,43621.680548,20123.763507,46192.256945,21258.466625,34388.840334,15866.447640,NaN,...,142734.07396,NaN,52434.225472,121969.841152,NaN,69163.765317,NaN,NaN,NaN,NaN
4,1,1,2,10346.119327,4582.132308,10251.260020,4562.389068,7159.829389,3218.099911,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,2,8,91050.846620,44161.868201,99445.065735,45985.967832,127746.359176,58973.611131,225514.73187,...,NaN,NaN,98468.806492,234895.205650,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
os.chdir(datapath)
survey_design_replication = pd.read_csv("weights_replication.txt", sep="\t")\
                            .rename(columns={'SEQN':'ID'})\
                            .set_index("ID")
survey_design_replication.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT2YR,WTMEC2YR,WTS_FFQ,BMXWT,BMIWT,CVDWTIM,WTSVOC2Y,...,WTSC2YR,WTSAF2YR,WTSA2YR,WTSCI2YR,WTSAU2YR,WTAL2YR,LBXDWT,WTSOG2YR,WTSC2YRA,WTSPC2YR
ID,,,,,,,,,,,,,,,,,,,,,
21005,3,2,39,5512.320949,5824.782465,4018.029553,137.6,NaN,NaN,NaN,...,18945.492755,14084.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21006,3,1,41,5422.140453,5564.039715,3838.164968,55.2,NaN,2.0,NaN,...,NaN,13081.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21007,3,2,35,39764.177412,40591.066325,26635.551187,47.9,NaN,2.0,NaN,...,NaN,NaN,132847.074292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21008,3,1,32,5599.499351,5696.750596,NaN,70.0,NaN,NaN,NaN,...,18529.060575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21009,3,2,31,97593.678977,97731.727244,67060.189883,103.1,NaN,NaN,182899.246183,...,276204.298055,NaN,NaN,NaN,186554.215404,NaN,NaN,NaN,NaN,NaN


In [6]:
# Divide replication weights by 2 to get 4 year weights
survey_design_replication.iloc[:,3:] = survey_design_replication.iloc[:,3:] / 2
survey_design_replication.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT2YR,WTMEC2YR,WTS_FFQ,BMXWT,BMIWT,CVDWTIM,WTSVOC2Y,...,WTSC2YR,WTSAF2YR,WTSA2YR,WTSCI2YR,WTSAU2YR,WTAL2YR,LBXDWT,WTSOG2YR,WTSC2YRA,WTSPC2YR
ID,,,,,,,,,,,,,,,,,,,,,
21005,3,2,39,2756.160474,2912.391233,2009.014777,68.80,NaN,NaN,NaN,...,9472.746377,7042.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21006,3,1,41,2711.070226,2782.019857,1919.082484,27.60,NaN,1.0,NaN,...,NaN,6540.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21007,3,2,35,19882.088706,20295.533162,13317.775593,23.95,NaN,1.0,NaN,...,NaN,NaN,66423.537146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21008,3,1,32,2799.749676,2848.375298,NaN,35.00,NaN,NaN,NaN,...,9264.530288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21009,3,2,31,48796.839489,48865.863622,33530.094942,51.55,NaN,NaN,91449.623091,...,138102.149027,NaN,NaN,NaN,93277.107702,NaN,NaN,NaN,NaN,NaN


In [7]:
var_description.head()

,tab_name,tab_desc,series,module,var_desc,analyzable,binary_ref_group,comment_var,version_date,is_comment,...,mexican_pct,black_N,black_pct,other_hispanic_N,other_hispanic_pct,other_N,other_pct,series_num,category,sub_category
var,,,,,,,,,,,,,,,,,,,,,
LBXHBC,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis B core antibody,1,2.0,NaN,2009-11-13,0.0,...,0.271533,7304.0,0.244698,1200.0,0.040202,1157.0,0.038762,3,viral infection,NaN
LBDHBG,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis B surface antigen,1,2.0,NaN,2009-11-13,0.0,...,0.271533,7304.0,0.244698,1200.0,0.040202,1157.0,0.038762,3,viral infection,NaN
LBDHCV,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis C antibody (confirmed),1,2.0,NaN,2009-11-13,0.0,...,0.271826,7269.0,0.244361,1198.0,0.040273,1153.0,0.038760,3,viral infection,NaN
LBDHD,l02_c,"Hepatitis A, B, C and D",2003-2004,laboratory,Hepatitis D (anti-HDV),1,2.0,NaN,2009-11-13,0.0,...,0.271533,7304.0,0.244698,1200.0,0.040202,1157.0,0.038762,3,viral infection,NaN
LBXHBS,l02hbs_c,Hepatitis B Surface Antibody,2003-2004,laboratory,Hepatitis B Surface Antibody,1,2.0,NaN,2009-11-13,0.0,...,0.273572,7879.0,0.247472,1317.0,0.041366,1272.0,0.039952,3,viral infection,NaN


In [8]:
os.chdir(datapath)
# These files map variables to their correct weights, and were compiled by reading throught the NHANES codebook
var_weights = pd.read_csv("VarWeights.csv")
var_weights.head()

,variable_name,discovery,replication
0,99999,WTMEC4YR,WTMEC2YR
1,ACETAMINOPHEN__CODEINE,WTMEC4YR,WTMEC2YR
2,ACETAMINOPHEN__CODEINE_PHOSPHATE,WTMEC4YR,WTMEC2YR
3,ACETAMINOPHEN__HYDROCODONE,WTMEC4YR,WTMEC2YR
4,ACETAMINOPHEN__HYDROCODONE_BITARTRATE,WTMEC4YR,WTMEC2YR


In [9]:
# Convert the data to two dictionaries for convenience
weights_discovery   = var_weights.set_index('variable_name')['discovery'].to_dict()
weights_replication = var_weights.set_index('variable_name')['replication'].to_dict()

## Define phenotypes and covariates


In [10]:
phenotype = ["LBXSCRINV","URXUCR","LBXSCR","LBXSATSI","LBXSAL","URXUMASI","URXUMA","LBXSAPSI","LBXSASSI","LBXSC3SI",
             "LBXSBU","LBXBAP","LBXCPSI","LBXCRP","LBXSCLSI","LBXSCH","LBDHDL","LBDLDL","LBXFER","LBXSGTSI","LBXSGB",
             "LBXGLU","LBXGH","LBXHCY","LBXSIR","LBXSLDSI","LBXMMA","LBXSOSSI","LBXSPH","LBXSKSI","LBXEPP",	
             "LBXSNASI","LBXTIB","LBXSTB","LBXSCA","LBXSTP","LBDPCT","LBXSTR","LBXSUA","LBDBANO","LBXBAPCT",
             "LBDEONO","LBXEOPCT","LBXHCT","LBXHGB","LBDLYMNO","LBXMCHSI","LBXLYPCT","LBXMCVSI","LBXMPSI","LBDMONO",
             "LBXMOPCT","LBXPLTSI","LBXRBCSI","LBXRDW","LBDNENO","LBXNEPCT","LBXIRN"]
for v in phenotype:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
        
covariates = ["black", "mexican", "other_hispanic", "other_eth", "SES_LEVEL", "RIDAGEYR", "SDDSRVYR","BMXBMI"]

	LBXSCRINV = 1/Creatinine (mg/dL)
	URXUCR = Creatinine, urine (mg/dL)
	LBXSCR = Creatinine (mg/dL)
	LBXSATSI = ALT (U/L)
	LBXSAL = Albumin (g/dL)
	URXUMASI = Albumin, urine (mg/L) SI
	URXUMA = Albumin, urine (ug/mL)
	LBXSAPSI = Alkaline phosphotase (U/L)
	LBXSASSI = AST (U/L)
	LBXSC3SI = Bicarbonate (mmol/L)
	LBXSBU = Blood urea nitrogen (mg/dL)
	LBXBAP = Bone alkaline phosphotase (ug/L)
	LBXCPSI = C-peptide: SI(nmol/L)
	LBXCRP = C-reactive protein(mg/dL)
	LBXSCLSI = Chloride (mmol/L)
	LBXSCH = Cholesterol, total (mg/dL)
	LBDHDL = Direct HDL-Cholesterol (mg/dL)
	LBDLDL = LDL-cholesterol (mg/dL)
	LBXFER = Ferritin (ng/mL)
	LBXSGTSI = GGT (U/L)
	LBXSGB = Globulin (g/dL)
	LBXGLU = Glucose, plasma (mg/dL)
	LBXGH = Glycohemoglobin (%)
	LBXHCY = Homocysteine (umol/L)
	LBXSIR = Iron (ug/dL)
	LBXSLDSI = LDH (U/L)
	LBXMMA = Methylmalonic acid (umol/L)
	LBXSOSSI = Osmolality (mOsml/L)
	LBXSPH = Phosphorus (mg/dL)
	LBXSKSI = Potassium (mmol/L)
	LBXEPP = Protoporphyrin (ug/dL RBC)
	LBXSNASI = Sodi

## Keep only adults

In [11]:
keep_adults = nhanes['RIDAGEYR'] >= 18
nhanes      = nhanes.loc[keep_adults]
nhanes.head()

,RIDAGEYR,female,male,black,mexican,other_hispanic,other_eth,SES_LEVEL,education,WTMEC2YR,...,MORTSTAT,MORTSOURCE_NDI,MORTSOURCE_SSA,PERMTH_INT,PERMTH_EXM,CAUSEAVL,UCODE,DIABETES,HYPERTEN,HIPFRACT
ID,,,,,,,,,,,,,,,,,,,,,
2,77,0,1,0,0,0,0,2.0,2.0,28325.384898,...,0.0,NaN,NaN,91.0,90.0,NaN,NaN,NaN,NaN,NaN
5,49,0,1,0,0,0,0,2.0,2.0,99445.065735,...,0.0,NaN,NaN,74.0,74.0,NaN,NaN,NaN,NaN,NaN
6,19,1,0,0,0,0,1,0.0,2.0,39656.600444,...,0.0,NaN,NaN,87.0,86.0,NaN,NaN,NaN,NaN,NaN
7,59,1,0,1,0,0,0,NaN,0.0,25525.423409,...,0.0,NaN,NaN,77.0,76.0,NaN,NaN,NaN,NaN,NaN
10,43,0,1,1,0,0,0,NaN,1.0,22445.808572,...,0.0,NaN,NaN,79.0,79.0,NaN,NaN,NaN,NaN,NaN


In [12]:
cl.describe.summarize(nhanes)

22,624 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Remove participants with missing values in covariates

In [13]:
nhanes = cl.modify.rowfilter_incomplete_obs(nhanes, only=covariates)

Running rowfilter_incomplete_obs
--------------------------------------------------------------------------------
Removed 3,687 of 22,624 observations (16.30%) due to NA values in any of 8 variables


In [14]:
cl.describe.summarize(nhanes)

18,937 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Keep only those variables that have weights

Also, some weight variables from the replication are not present, remove them as well

In [15]:
cols = survey_design_replication.columns
vals = set(weights_replication.values())

#Which survey weights are not present
weights_absent = []
for i in set(vals):
    if i not in cols:
        print(i + ' is not in cols')
        weights_absent.append(i)

#Which variables need those survey weights
remove_vars = []
for key, value in weights_replication.items():
    for i in weights_absent:
        if i == value: 
            remove_vars.append(key)

WTSPP2YR is not in cols
WTSTH2YR is not in cols
WTSPH2YR is not in cols


In [16]:
# Create a whole list with variables and remove the ones without weigts
keep_vars = list(weights_replication.keys())
keep_vars.extend(['SDDSRVYR', 'SDMVPSU', 'SDMVSTRA'])
keep_vars.extend(covariates)
keep_vars.extend(phenotype)
keep_vars.extend(['male', 'female'])
keep_vars = list(set(keep_vars))

for i in remove_vars:
    if i in keep_vars:
        print(i + ' is in keep vars')
        keep_vars.remove(i)
        

LBXTSH is in keep vars
URX24D is in keep vars
URX25T is in keep vars
URX4FP is in keep vars
URXACE is in keep vars
URXATZ is in keep vars
URXCB3 is in keep vars
URXCCC is in keep vars
URXCMH is in keep vars
URXCPM is in keep vars
URXDEE is in keep vars
URXDIZ is in keep vars
URXDPY is in keep vars
URXMET is in keep vars
URXOPM is in keep vars
URXP08 is in keep vars
URXPAR is in keep vars
URXTCC is in keep vars


In [17]:
# Keep only those vars in keep_vars
nhanes = cl.modify.colfilter(nhanes, only=keep_vars)

Running colfilter
--------------------------------------------------------------------------------
Keeping 1,069 of 1,190 variables:
	0 of 0 binary variables
	0 of 0 categorical variables
	1,068 of 1,189 continuous variables
	1 of 1 unknown variables


## Variable selection and clenup

Here we will:
- Drop any variables that are indeterminant according to the NHANES data dictionary
- Drop variables that don't have 4 year weights
- Drop any non-environmental exposures (examples physical fitness)
- Determine covariates and phenotypes
- From the remainder of the variables split them based on their type
- Manually inspect the definition of ambiguous variables and determine their type
- Merge binary into categorical variables


In [18]:
#### RECODING VARIABLES

# SMQ077 and DDB100 have Refused/Don't Know for "7" and "9" values.
# We will recode them as nan
nhanes = cl.modify.recode_values(nhanes, {7: np.nan, 9: np.nan}, only=['SMQ077', 'DBD100'])

Running recode_values
--------------------------------------------------------------------------------
Replaced 11 values from 18,937 observations in 2 variables


In [19]:
#### DROPING PHYSICAL FITNESS

# Droping physical fitness measurements, because they are not proxies for environmental exposures
phys_fitness_vars = ["CVDVOMAX","CVDESVO2","CVDS1HR","CVDS1SY","CVDS1DI","CVDS2HR","CVDS2SY","CVDS2DI","CVDR1HR","CVDR1SY","CVDR1DI","CVDR2HR","CVDR2SY","CVDR2DI","physical_activity"]
for v in phys_fitness_vars:
    print(f"\t{v} = {var_descr_dict[v]}")
nhanes = nhanes.drop(columns=phys_fitness_vars)

#Nikki also droped this unknown pesticide ('URXP08')

	CVDVOMAX = Predicted VO2max (ml/kg/min)
	CVDESVO2 = Estimated VO2max (ml/kg/min)
	CVDS1HR = Stage 1 heart rate (per min)
	CVDS1SY = Stage 1 systolic BP (mm Hg)
	CVDS1DI = Stage 1 diastolic BP (mm Hg)
	CVDS2HR = Stage 2 heart rate (per min)
	CVDS2SY = Stage 2 systolic BP (mm Hg)
	CVDS2DI = Stage 2 diastolic BP (mm Hg)
	CVDR1HR = Recovery 1 heart rate (per min)
	CVDR1SY = Recovery 1 systolic BP (mm Hg)
	CVDR1DI = Recovery 1 diastolic BP (mm Hg)
	CVDR2HR = Recovery 2 heart rate (per min)
	CVDR2SY = Recovery 2 systolic BP (mm Hg)
	CVDR2DI = Recovery 2 diastolic BP (mm Hg)
	physical_activity = Physical Activity (MET-based rank)


In [20]:
#### DROPING INDETERMINATE VARIABLES

indeterminent_vars = ["house_type","hepa","hepb","house_age","current_past_smoking","house_age","DUQ130","DMDMARTL","income"]
print("Removing variables with indeterminate meanings:")
for v in indeterminent_vars:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
nhanes = nhanes.drop(columns=indeterminent_vars)

Removing variables with indeterminate meanings:
	house_type = house type
	hepa = hepatitis a
	hepb = hepatitis b
	house_age = house age
	current_past_smoking = Current or Past Cigarette Smoker?
	house_age = house age
	DUQ130 = #days used needle for street drugs/year


In [21]:
cl.describe.summarize(nhanes)

18,937 observations of 1,046 variables
	0 Binary Variables
	0 Categorical Variables
	1,045 Continuous Variables
	1 Unknown-Type Variables



## Adjust lipids if on statins

In [22]:
#Adjust for statins (LDL=LDL/0.7, TC=TC/0.8) (2)
statins   = ["ATORVASTATIN_CALCIUM", "SIMVASTATIN", "PRAVASTATIN_SODIUM", "FLUVASTATIN_SODIUM"]
for v in statins:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
nhanes.loc[(nhanes[statins].sum(axis=1) > 0), 'LBDLDL'] = nhanes.loc[(nhanes[statins].sum(axis=1) > 0), 'LBDLDL']/0.7

	ATORVASTATIN_CALCIUM = ATORVASTATIN_CALCIUM
	SIMVASTATIN = SIMVASTATIN
	PRAVASTATIN_SODIUM = PRAVASTATIN_SODIUM
	FLUVASTATIN_SODIUM = FLUVASTATIN_SODIUM


## Separate discovery and replication, and males and females

In [23]:
discovery   = (nhanes['SDDSRVYR']==1) | (nhanes['SDDSRVYR']==2)
replication = (nhanes['SDDSRVYR']==3) | (nhanes['SDDSRVYR']==4)

nhanes_discovery   = nhanes.loc[discovery]
nhanes_replication = nhanes.loc[replication]

In [24]:
keep_discovery_females = nhanes_discovery.female == 1
keep_discovery_males   = nhanes_discovery.male   == 1

keep_replication_females = nhanes_replication.female == 1
keep_replication_males   = nhanes_replication.male   == 1

nhanes_discovery_females = nhanes_discovery.loc[keep_discovery_females]
nhanes_discovery_males   = nhanes_discovery.loc[keep_discovery_males]

nhanes_replication_females = nhanes_replication.loc[keep_replication_females]
nhanes_replication_males   = nhanes_replication.loc[keep_replication_males]

## Remove variables without weights

In [25]:
keep_discovery   = set(weights_discovery.keys()) | set(phenotype + covariates)
keep_replication = set(weights_replication.keys()) | set(phenotype + covariates)

nhanes_discovery_females = nhanes_discovery_females[[c for c in list(nhanes_discovery_females) if c in keep_discovery]]
nhanes_discovery_males = nhanes_discovery_males[[c for c in list(nhanes_discovery_males) if c in keep_discovery]]

nhanes_replication_females = nhanes_replication_females[[c for c in list(nhanes_replication_females) if c in keep_replication]]
nhanes_replication_males = nhanes_replication_males[[c for c in list(nhanes_replication_males) if c in keep_replication]]

In [26]:
dfs = [nhanes_discovery_females, nhanes_discovery_males, nhanes_replication_females, nhanes_replication_males]
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 1,044 variables
	0 Binary Variables
	0 Categorical Variables
	1,043 Continuous Variables
	1 Unknown-Type Variables

4,339 observations of 1,044 variables
	0 Binary Variables
	0 Categorical Variables
	1,043 Continuous Variables
	1 Unknown-Type Variables

5,123 observations of 1,044 variables
	0 Binary Variables
	0 Categorical Variables
	1,043 Continuous Variables
	1 Unknown-Type Variables

4,751 observations of 1,044 variables
	0 Binary Variables
	0 Categorical Variables
	1,043 Continuous Variables
	1 Unknown-Type Variables



# Quality control

## Categorize variables

It will also remove all NA variables

In [27]:
for i in range(len(dfs)):
    dfs[i] = cl.modify.categorize(dfs[i])

Running categorize
--------------------------------------------------------------------------------
54 of 1,044 variables (5.17%) are classified as constant (1 unique value).
305 of 1,044 variables (29.21%) are classified as binary (2 unique values).
37 of 1,044 variables (3.54%) are classified as categorical (3 to 6 unique values).
392 of 1,044 variables (37.55%) are classified as continuous (>= 15 unique values).
208 of 1,044 variables (19.92%) were dropped.
	208 variables had zero unique values (all NA).
48 of 1,044 variables (4.60%) were not categorized and need to be set manually.
	47 variables had between 6 and 15 unique values
	1 variables had >= 15 values but couldn't be converted to continuous (numeric) values
Running categorize
--------------------------------------------------------------------------------
63 of 1,044 variables (6.03%) are classified as constant (1 unique value).
278 of 1,044 variables (26.63%) are classified as binary (2 unique values).
40 of 1,044 variable

## Remove constant variables

In [28]:
# Remove constant variables (Except male and female which should be constant as well)
remove_constants = []
for i in range(len(dfs)):
    var_types    = cl.describe.get_types(dfs[i])
    var_constant = var_types[var_types == 'constant'].index
    remove_constants.extend(var_constant)
    
remove_constants = set(remove_constants) # Get unique elements

for i in ['female', 'male']: # Remove male and female from list
    remove_constants.remove(i)

for v in list(remove_constants):
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")

	IRON_Unknown = IRON_Unknown
	PHOSPHORUS_Unknown = PHOSPHORUS_Unknown
	occupation_repair = occupation_repair
	URXOMO = O-methoate (ug/L)
	URXPPX = 2-isopropoxyphenol (ug/L)
	SMD100FL = Filter type
	CONJUGATED_ESTROGENS = CONJUGATED_ESTROGENS
	LBXV4E = Blood 1,1,2,2-Tetrachloroethane (ng/mL)
	occupation_farm = occupation_farm
	MAGNESIUM_PPM = MAGNESIUM_PPM
	BETA_CAROTENE_mcg = BETA_CAROTENE_mcg
	RIBOFLAVIN_Unknown = RIBOFLAVIN_Unknown
	OTHER_FATTY_ACIDS_NA = OTHER_FATTY_ACIDS_NA
	COPPER_Trace = COPPER_Trace
	URXHLS = Halosulfuron (ug/L)
	CYSTINE_mg = CYSTINE_mg
	industry_transport = industry_transport
	VITAMIN_B_6_Unknown = VITAMIN_B_6_Unknown
	prostate_cancer_self_report = prostate_cancer_self_report
	industry_other = industry_other
	VITAMIN_A_mg = VITAMIN_A_mg
	SMQ230 = Do you now use chewing tobacco
	industry_agriculture = industry_agriculture
	ETHINYL_ESTRADIOL__NORETHINDRONE = ETHINYL_ESTRADIOL__NORETHINDRONE
	CALCIUM_Unknown = CALCIUM_Unknown
	VITAMIN_A_mcg = VITAMIN_A_mcg
	larynx

In [29]:
#Remove columns in datasets
for i in range(len(dfs)):
    remove_this = list(set(remove_constants) & set(dfs[i].columns))
    dfs[i]      = dfs[i].drop(columns=remove_this)

In [30]:
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 742 variables
	278 Binary Variables
	34 Categorical Variables
	391 Continuous Variables
	37 Unknown-Type Variables

4,339 observations of 711 variables
	262 Binary Variables
	32 Categorical Variables
	385 Continuous Variables
	30 Unknown-Type Variables

5,123 observations of 784 variables
	270 Binary Variables
	42 Categorical Variables
	426 Continuous Variables
	44 Unknown-Type Variables

4,751 observations of 754 variables
	260 Binary Variables
	38 Categorical Variables
	421 Continuous Variables
	33 Unknown-Type Variables



## Examine unknown variables

In all datasets the variable L_GLUTAMINE_gm is incorrectly assigned as categorical, but it is continuous

In [31]:
v = "L_GLUTAMINE_gm"
print(f"\t{v} = {var_descr_dict[v]}\n")
for i in range(len(dfs)):
    if v in dfs[i].columns:
        dfs[i] = cl.modify.make_continuous(dfs[i], only=[v])

	L_GLUTAMINE_gm = L_GLUTAMINE_gm



In [32]:
text  = ['Discovery females', 'Discovery males', 'Replication females', 'Replication males']
for i in range(len(dfs)):
    var_types   = cl.describe.get_types(dfs[i])
    var_unknown = var_types[var_types == 'unknown'].index
    print('Unknown variables in ' + text[i] + ':')
    for v in list(var_unknown):
        if v in var_descr_dict:
            print(f"\t{v} = {var_descr_dict[v]}")


Unknown variables in Discovery females:
	URXUBE = Beryllium, urine (ng/mL)
	URXUPT = Platinum, urine (ng/mL)
	LBDBANO = Basophils number
	quantity_drink_per_day = drink per day
	SMD220 = Age started chewing tobacco regularly
	SMD190 = Age started using snuff regularly
	SMD160 = Age started cigar smoking regularly
	SMD130 = Age started pipe smoking regularly
	DRD350BQ = # of times crabs eaten in past 30 days
	DRD350FQ = # of times oysters eaten in past 30 days
	DRD350HQ = # of times shrimp eaten in past 30 days
	DRD370AQ = # of times breaded fish products eaten
	DRD370DQ = # of times catfish eaten in past 30 days
	DRD370EQ = # of times cod eaten in past 30 days
	DRD370FQ = # of times flatfish eaten past 30 days
	DRD370IQ = # of times perch eaten in past 30 days
	DRD370KQ = # of times pollock eaten in past 30 days
	DRD370MQ = # of times salmon eaten in past 30 days
	DRD370NQ = # of times sardines eaten past 30 days
	DRD370TQ = # of times other fish eaten past 30 days
	DRD370UQ = # of tim

In all datasets all unknown variables are continuous, except area which is categorical and it's at the end of the list

In [33]:
for i in range(len(dfs)):
    var_types   = cl.describe.get_types(dfs[i])
    var_unknown = var_types[var_types == 'unknown'].index
    dfs[i] = cl.modify.make_continuous(dfs[i], only=var_unknown[:-1])
    dfs[i] = cl.modify.make_categorical(dfs[i], only=var_unknown[-1])

Running make_continuous
--------------------------------------------------------------------------------
Set 36 of 742 variable(s) as continuous, each with 4,724 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 1 of 742 variable(s) as categorical, each with 4,724 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 29 of 711 variable(s) as continuous, each with 4,339 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 1 of 711 variable(s) as categorical, each with 4,339 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 43 of 784 variable(s) as continuous, each with 5,123 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 1 of 784 vari

In [34]:
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 742 variables
	278 Binary Variables
	35 Categorical Variables
	427 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 711 variables
	262 Binary Variables
	33 Categorical Variables
	414 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 784 variables
	270 Binary Variables
	43 Categorical Variables
	469 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 754 variables
	260 Binary Variables
	39 Categorical Variables
	453 Continuous Variables
	0 Unknown-Type Variables



## Remove phenotypes that were dropped

Because of the QC, some phenotypes might have been deleted

In [35]:
remove_phenos = []
for i in phenotype:
    for l in range(len(dfs)):
        cols = dfs[l].columns
        if i not in cols:
            print(i + ' was deleted in at least one dataset')
            remove_phenos.append(i)

# Remove the deleted phenos from all datasets
for i in remove_phenos:
    phenotype.remove(i)
    for l in range(len(dfs)):
        if i in dfs[l].columns:
            dfs[l] = dfs[l].drop(columns=i)

LBXFER was deleted in at least one dataset
LBXEPP was deleted in at least one dataset
LBXTIB was deleted in at least one dataset
LBDPCT was deleted in at least one dataset
LBXIRN was deleted in at least one dataset


## QC

We will remove variables that have less than 200 non-nan values

In [36]:
for i in range(len(dfs)):
    dfs[i] = cl.modify.colfilter_min_n(dfs[i], skip=phenotype + covariates + ['male', 'female'])

Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 273 of 278 binary variables
	Removed 29 (10.62%) tested binary variables which had less than 200 non-null values.
Testing 34 of 35 categorical variables
	Removed 18 (52.94%) tested categorical variables which had less than 200 non-null values.
Testing 367 of 422 continuous variables
	Removed 17 (4.63%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 257 of 262 binary variables
	Removed 23 (8.95%) tested binary variables which had less than 200 non-null values.
Testing 32 of 33 categorical variables
	Removed 17 (53.12%) tested categorical variables which had less than 200 non-null values.
Testing 354 of 409 continuous variables
	Removed 12 (3.39%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
-----

## Filtering

In [37]:
# 200 non-na samples
for i in range(len(dfs)):
    dfs[i] = cl.modify.colfilter_min_n(dfs[i])

Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 249 of 249 binary variables
	Removed 0 (0.00%) tested binary variables which had less than 200 non-null values.
Testing 17 of 17 categorical variables
	Removed 0 (0.00%) tested categorical variables which had less than 200 non-null values.
Testing 405 of 405 continuous variables
	Removed 0 (0.00%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 239 of 239 binary variables
	Removed 0 (0.00%) tested binary variables which had less than 200 non-null values.
Testing 16 of 16 categorical variables
	Removed 0 (0.00%) tested categorical variables which had less than 200 non-null values.
Testing 397 of 397 continuous variables
	Removed 0 (0.00%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------

In [38]:
# 200 samples per category
for i in range(len(dfs)):
    dfs[i] = cl.modify.colfilter_min_cat_n(dfs[i], skip=[c for c in covariates + phenotype + ['male', 'female'] if c in dfs[i].columns] )

Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 244 of 249 binary variables
	Removed 185 (75.82%) tested binary variables which had a category with less than 200 values.
Testing 16 of 17 categorical variables
	Removed 12 (75.00%) tested categorical variables which had a category with less than 200 values.
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 234 of 239 binary variables
	Removed 182 (77.78%) tested binary variables which had a category with less than 200 values.
Testing 15 of 16 categorical variables
	Removed 10 (66.67%) tested categorical variables which had a category with less than 200 values.
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 239 of 244 binary variables
	Removed 178 (74.48%) tested binary variables which had a category with less than 200 values.
Testing 2

In [39]:
# 90percent zero filter
for i in range(len(dfs)):
    dfs[i] = cl.modify.colfilter_percent_zero(dfs[i])

Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 405 of 405 continuous variables
	Removed 13 (3.21%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 397 of 397 continuous variables
	Removed 12 (3.02%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 432 of 432 continuous variables
	Removed 12 (2.78%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 430 of 430 continuous variables
	Removed 14 (3.26%) tested continuous variab

In [40]:
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 463 variables
	64 Binary Variables
	5 Categorical Variables
	392 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 450 variables
	57 Binary Variables
	6 Categorical Variables
	385 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 493 variables
	66 Binary Variables
	5 Categorical Variables
	420 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 486 variables
	62 Binary Variables
	6 Categorical Variables
	416 Continuous Variables
	0 Unknown-Type Variables



In [41]:
# Take note of which variables were differently typed in each dataset
print("Correcting differences in variable types between discovery and replication")
# Merge current type series
dtypes = pd.DataFrame({'discovery_females':cl.describe.get_types(dfs[0]),
                       'discovery_males':cl.describe.get_types(dfs[1]),
                       'replication_females':cl.describe.get_types(dfs[2]),
                       'replication_males':cl.describe.get_types(dfs[3]),
                       })

for i,l in dtypes.iterrows():
    m = set(l)
    m = {x for x in m if x==x}
    if len(m) > 1:
        print('There is no agreement in here', m )

# There are no differences

Correcting differences in variable types between discovery and replication


## Keeping variables that are in all datasets

In [42]:
all_vars = set(list(dfs[0])) & set(list(dfs[1])) & set(list(dfs[2])) & set(list(dfs[3]))
for i in range(len(dfs)):
    dfs[i] = dfs[i][all_vars]
    
print(f"{len(all_vars)} variables in common")

393 variables in common


## Remove categorical variables

In [43]:
# Categorical variables don't get a beta and SE estimation needed for the sex difference test
# Make sure to not remove categorical that are covariates
for i in range(len(dfs)):
    var_types = cl.describe.get_types(dfs[i])
    var_cat   = list(var_types[var_types == 'categorical'].index)
    for l in covariates:
        if l in var_cat:
            var_cat.remove(l)
    dfs[i] = dfs[i].drop(columns=var_cat)

## Normalize variables

In [44]:
# Merge them to normalize across sexes while keeping the previous variable categories
var_types = cl.describe.get_types(dfs[0])
var_continuous  = list(var_types[var_types == 'continuous'].index)
var_binary      = var_types[var_types == 'binary'].index
var_categorical = var_types[var_types == 'categorical'].index
var_constant    = var_types[var_types == 'constant'].index

#nhanes = cl.modify.merge_observations(nhanes_females, nhanes_males)
nhanes = pd.concat([dfs[0], dfs[1], dfs[2], dfs[3]])

cl.describe.summarize(nhanes)

18,937 observations of 389 variables
	44 Binary Variables
	1 Categorical Variables
	344 Continuous Variables
	0 Unknown-Type Variables



In [45]:
# Normalize only continuous that are not covariates
for i in covariates:
    if i in var_continuous:
        var_continuous.remove(i)

# Get the columns that are not in the var_continuous list
cols = list(nhanes.columns)
for i in var_continuous:
    cols.remove(i)

In [46]:
from sklearn.preprocessing import normalize

nhanes_c = nhanes[var_continuous] # nhanes continuous without covariates
nhanes_e = nhanes[cols] # nhanes everything else

nhanes_c = (nhanes_c - nhanes_c.min()) / (nhanes_c.max() - nhanes_c.min()) # Normalizing with pandas ignoring NaN
nhanes_c

,LBX156,URXP20,URXP11,LBXTCD,URXEQU,LBXLYC,SMD430,URXP14,URXMZP,LBXVBM,...,LBXF03,LBX128,LBXWCM,DR1TZINC,URXP04,supplement_count,LBX028,URXUMO,LBXCRP,LBXLUZ
ID,,,,,,,,,,,,,,,,,,,,,
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.024053,NaN,0.052632,NaN,0.06116,0.001690,NaN
15,NaN,NaN,0.000487,NaN,NaN,NaN,0.179487,NaN,0.001309,NaN,...,NaN,NaN,NaN,0.041449,0.013588,0.000000,NaN,NaN,0.006759,NaN
16,NaN,NaN,NaN,NaN,0.000100,NaN,NaN,NaN,0.000380,NaN,...,NaN,NaN,NaN,0.022785,NaN,0.263158,NaN,NaN,0.001014,NaN
20,NaN,NaN,0.000487,NaN,0.000362,NaN,NaN,0.0,0.000319,NaN,...,NaN,NaN,NaN,0.084413,0.001271,0.052632,NaN,NaN,0.022305,NaN
24,0.017186,NaN,NaN,0.024006,NaN,NaN,1.000000,NaN,NaN,NaN,...,0.057686,0.164244,0.000203,0.018805,NaN,0.000000,0.062649,NaN,0.002366,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41461,NaN,NaN,NaN,NaN,0.000723,0.032699,NaN,NaN,0.001322,0.064083,...,NaN,NaN,0.024713,0.024827,NaN,0.000000,NaN,NaN,0.001014,0.077584
41462,NaN,NaN,NaN,NaN,NaN,0.326897,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.047207,NaN,0.105263,NaN,0.02364,0.001352,0.275972
41469,NaN,NaN,NaN,NaN,0.000368,0.489935,NaN,NaN,0.005613,0.008186,...,NaN,NaN,0.013944,0.030233,NaN,0.000000,NaN,NaN,0.000000,0.128952


In [47]:
# Merge back
nhanes = cl.modify.merge_variables(nhanes_e, nhanes_c, how='inner')
nhanes

Running merge_variables
--------------------------------------------------------------------------------
inner Merge:
	left = 18,937 observations of 50 variables
	right = 18,937 observations of 339 variables
Kept 18,937 observations of 389 variables.


,drink_five_per_day,DRD360,DRD370B,female,LBXHA,cigarette_smoking,black,any_ht,LBXHBC,SMQ020,...,LBXF03,LBX128,LBXWCM,DR1TZINC,URXP04,supplement_count,LBX028,URXUMO,LBXCRP,LBXLUZ
ID,,,,,,,,,,,,,,,,,,,,,
6,NaN,1.0,1.0,1,1.0,NaN,0,0.0,0.0,NaN,...,NaN,NaN,NaN,0.024053,NaN,0.052632,NaN,0.06116,0.001690,NaN
15,0.0,1.0,1.0,1,0.0,1.0,0,0.0,0.0,1.0,...,NaN,NaN,NaN,0.041449,0.013588,0.000000,NaN,NaN,0.006759,NaN
16,NaN,1.0,0.0,1,1.0,NaN,1,0.0,0.0,0.0,...,NaN,NaN,NaN,0.022785,NaN,0.263158,NaN,NaN,0.001014,NaN
20,0.0,1.0,1.0,1,1.0,0.0,0,0.0,0.0,1.0,...,NaN,NaN,NaN,0.084413,0.001271,0.052632,NaN,NaN,0.022305,NaN
24,0.0,1.0,1.0,1,1.0,1.0,0,0.0,0.0,1.0,...,0.057686,0.164244,0.000203,0.018805,NaN,0.000000,0.062649,NaN,0.002366,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41461,0.0,0.0,NaN,0,1.0,1.0,0,1.0,0.0,0.0,...,NaN,NaN,0.024713,0.024827,NaN,0.000000,NaN,NaN,0.001014,0.077584
41462,0.0,1.0,1.0,0,1.0,1.0,0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.047207,NaN,0.105263,NaN,0.02364,0.001352,0.275972
41469,NaN,0.0,NaN,0,0.0,1.0,1,0.0,0.0,NaN,...,NaN,NaN,0.013944,0.030233,NaN,0.000000,NaN,NaN,0.000000,0.128952


In [48]:
discovery   = (nhanes['SDDSRVYR']==1) | (nhanes['SDDSRVYR']==2)
replication = (nhanes['SDDSRVYR']==3) | (nhanes['SDDSRVYR']==4)

nhanes_discovery   = nhanes.loc[discovery]
nhanes_replication = nhanes.loc[replication]

keep_discovery_females = nhanes_discovery.female == 1
keep_discovery_males   = nhanes_discovery.male   == 1

keep_replication_females = nhanes_replication.female == 1
keep_replication_males   = nhanes_replication.male   == 1

nhanes_discovery_females = nhanes_discovery.loc[keep_discovery_females]
nhanes_discovery_males   = nhanes_discovery.loc[keep_discovery_males]

nhanes_replication_females = nhanes_replication.loc[keep_replication_females]
nhanes_replication_males   = nhanes_replication.loc[keep_replication_males]

dfs = [nhanes_discovery_females, nhanes_discovery_males, nhanes_replication_females, nhanes_replication_males]

In [49]:
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 389 variables
	44 Binary Variables
	1 Categorical Variables
	344 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 389 variables
	44 Binary Variables
	1 Categorical Variables
	344 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 389 variables
	44 Binary Variables
	1 Categorical Variables
	344 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 389 variables
	44 Binary Variables
	1 Categorical Variables
	344 Continuous Variables
	0 Unknown-Type Variables



## Inspect variables

In [50]:
def inspect_variables(data, suffix=''):
    #### Plot variables to manually inspect them
    os.chdir(os.path.join(mainpath, 'Results', 'Plots'))
    
    var_types  = cl.describe.get_types(data)
    var_binary = var_types[var_types == 'binary'].index
    var_categorical = var_types[var_types == 'categorical'].index
    var_continuous  = var_types[var_types == 'continuous'].index
    
    vartypes = [var_binary, var_categorical, var_continuous, phenotype, covariates]
    names    = ['var_binary', 'var_categorical', 'var_continuous', 'phenotype', 'covariates']
    
    for i in range(len(vartypes)):
        outname = names[i] + suffix + '.pdf'
        cl.plot.distributions(data,
                              filename=outname,
                              continuous_kind='count',
                              nrows=4,
                              ncols=3,
                              quality='medium',
                              variables=list(vartypes[i]))

In [51]:
text = ['_d_f', '_d_m', '_r_f', '_r_m']
for i in range(len(dfs)):
    inspect_variables(dfs[i], text[i])

Generating a 4 page PDF for 44 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or 

Generating a 1 page PDF for 1 variables
Generating a 29 page PDF for 344 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 5 page PDF for 53 variables


...

/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 5 page PDF for 53 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)


Generating a 1 page PDF for 8 variables


/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/distributions.py:2551: FutureWarning: `distplot` is a deprecated function and will be removed in a future version. Please adapt your code to use either `displot` (a figure-level function with similar flexibility) or `histplot` (an axes-level function for histograms).
  warnings.warn(msg, FutureWarning)
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  FutureWarning
/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/seaborn/_decorators.py:43: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit ke

## Remove sex and unnecesary variables

In [52]:
remove_vars = ['male', 'female', 'white'] # The last one is not in the replication weight
for i in range(len(dfs)):
    dfs[i] = dfs[i].drop(columns=remove_vars)

In [53]:
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 386 variables
	43 Binary Variables
	1 Categorical Variables
	342 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 386 variables
	43 Binary Variables
	1 Categorical Variables
	342 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 386 variables
	43 Binary Variables
	1 Categorical Variables
	342 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 386 variables
	43 Binary Variables
	1 Categorical Variables
	342 Continuous Variables
	0 Unknown-Type Variables



## Save cleaned data

In [54]:
os.chdir(os.path.join(mainpath, 'Results'))
savef = ['Discovery_Females', 'Discovery_Males', 'Replication_Females', 'Replication_Males']
for i in range(len(dfs)):
    dfs[i].to_csv('CleanData_' + savef[i] + '.csv' )